In [1]:
import pandas as pd

In [10]:
## Example Daily Weather dataset
data = {
    "Date": pd.date_range("2025-01-01",periods=120),
    "Tmin": [2,3,4,6,8,10]*20,
    "Tmax": [8,10,12,14,16,18]*20
}

df = pd.DataFrame(data)
print(df.head())

        Date  Tmin  Tmax
0 2025-01-01     2     8
1 2025-01-02     3    10
2 2025-01-03     4    12
3 2025-01-04     6    14
4 2025-01-05     8    16


In [12]:
## parameters
Tbase = 5 # °C
chill_requirements = 1000 # Chill units ( example for apple)
gdd_thresholds = {
    "bud_break" : 100,
    "flowerng" : 350,
    "fruit_set": 600,
    "harvest" : 1300
}


# Step 1 : Calculate Chill units 
def chill_units(Tmin,Tmax):
    Tavg = (Tmin + Tmax)/2
    if 0 <= Tavg <= 7:
        return 1
    elif 7 < Tavg <= 15:
        return 0.5
    elif Tavg > 20:
        return -0.5
    else:
        return 0


df["Chill"] = df.apply(lambda row: chill_units(row["Tmin"],row["Tmax"]),axis=1)
df["Cummulative_Chill"] = df["Chill"].cumsum()


## Step2 : Start GDD accumulation after chilling requirements is met
df["GDD"] = 0
cummulative_gdd = 0
chill_met = False
stages = {}


for i , row in df.iterrows():
    if not chill_met:
        if row["Cummulative_Chill"] >= chill_requirements:
            chill_met = True
        continue

    # Calculate Daily GDD
    Tavg = (row["Tmax"] + row["Tmin"])/2
    gdd = max(0,Tavg - Tbase)
    cummulative_gdd += gdd
    df.at[i,"GDD"] = cummulative_gdd

    # Record when threshold are passed
    for stage, threshold in gdd_thresholds.items():
        if stage not in stages and cummulative_gdd >= threshold:
            stages[stage] = row["Date"]


# Output 
print("Phanelogical Stages (Predicted) : ")
for stage , date in stages.items():
    print(f"{stage} : {date.date()}")

# print(df.head())

Phanelogical Stages (Predicted) : 


In [13]:
import pandas as pd

# -----------------------------
# Example: Replace this with your real weather data (CSV, API, etc.)
# Date, Tmin, Tmax
data = {
    "Date": pd.date_range("2025-01-01", periods=120),
    "Tmin": [2, 3, 4, 6, 8, 10] * 20,  # dummy repeating data
    "Tmax": [8, 10, 12, 14, 16, 18] * 20
}
df = pd.DataFrame(data)

# -----------------------------
# Parameters (adjust for your apple variety & location)
Tbase = 5  # °C (base temperature for apple GDD)
chill_requirement = 1000  # chill units needed to break dormancy
gdd_thresholds = {
    "bud_break": 100,
    "flowering": 350,
    "fruit_set": 600,
    "harvest": 1300
}

# -----------------------------
# Step 1: Calculate Chill Units (simplified version)
def chill_units(Tmin, Tmax):
    Tavg = (Tmin + Tmax) / 2
    if 0 <= Tavg <= 7:
        return 1
    elif 7 < Tavg <= 15:
        return 0.5
    elif Tavg > 20:
        return -0.5  # warm temps reduce chilling
    else:
        return 0

df["Chill"] = df.apply(lambda row: chill_units(row["Tmin"], row["Tmax"]), axis=1)
df["Cumulative_Chill"] = df["Chill"].cumsum()

# -----------------------------
# Step 2: Start GDD after chilling requirement is met
df["Daily_GDD"] = 0
df["Cumulative_GDD"] = 0
chill_met = False
cumulative_gdd = 0
stages = {}

for i, row in df.iterrows():
    if not chill_met:
        if row["Cumulative_Chill"] >= chill_requirement:
            chill_met = True
        continue  # keep waiting until chilling is complete
    
    # Calculate GDD
    Tavg = (row["Tmin"] + row["Tmax"]) / 2
    gdd = max(0, Tavg - Tbase)
    cumulative_gdd += gdd
    df.at[i, "Daily_GDD"] = gdd
    df.at[i, "Cumulative_GDD"] = cumulative_gdd
    
    # Record when thresholds are crossed
    for stage, threshold in gdd_thresholds.items():
        if stage not in stages and cumulative_gdd >= threshold:
            stages[stage] = row["Date"]

# -----------------------------
# Results
print("📅 Predicted Phenological Stages for Apple:")
for stage, date in stages.items():
    print(f"{stage}: {date.date()}")

print("\n📊 Sample Data (first 20 days):")
print(df.head(20))


📅 Predicted Phenological Stages for Apple:

📊 Sample Data (first 20 days):
         Date  Tmin  Tmax  Chill  Cumulative_Chill  Daily_GDD  Cumulative_GDD
0  2025-01-01     2     8    1.0               1.0          0               0
1  2025-01-02     3    10    1.0               2.0          0               0
2  2025-01-03     4    12    0.5               2.5          0               0
3  2025-01-04     6    14    0.5               3.0          0               0
4  2025-01-05     8    16    0.5               3.5          0               0
5  2025-01-06    10    18    0.5               4.0          0               0
6  2025-01-07     2     8    1.0               5.0          0               0
7  2025-01-08     3    10    1.0               6.0          0               0
8  2025-01-09     4    12    0.5               6.5          0               0
9  2025-01-10     6    14    0.5               7.0          0               0
10 2025-01-11     8    16    0.5               7.5          0      